# Project 27 — Colab quickstart
Notebook gọi lại code repo, không triển khai training thứ hai. Chọn Runtime → Change runtime type → GPU. Repo riêng tư: tải source ZIP từ GitHub Release bằng tài khoản của anh rồi upload ở cell dưới. Không dán token vào notebook.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import os

uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
destination = Path("/content/project27")
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    for item in bundle.infolist():
        assert (destination / item.filename).resolve().is_relative_to(destination.resolve())
    bundle.extractall(destination)
roots = [p.parent for p in destination.rglob("main.py") if (p.parent / "configs/base.yaml").exists()]
assert len(roots) == 1
os.chdir(roots[0])
print(Path.cwd())

In [ ]:
%pip install -r requirements-dev.txt

In [ ]:
import torch

print(torch.__version__, torch.cuda.is_available())
!python -m pytest -q

## Lưu checkpoint vào Google Drive
Cấu hình output ở Drive, dataset cache ở /content. Không đổi config khi resume một run cũ. Nếu Drive đã chứa kết quả khác backend hoặc hyperparameter, dùng thư mục mới.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
output_dir = "/content/drive/MyDrive/project27-fresh/outputs"

In [ ]:
!python main.py --config configs/iid.yaml --seed 42 --device cuda --output-dir "$output_dir" --resume

## Chạy toàn study
15 run, 24.300.000 lượt ảnh train. Checkpoint/resume tại mỗi round. Không dùng test để điều chỉnh cấu hình. Notebook này chưa được chạy trực tiếp trên Colab trong lần bàn giao.


In [ ]:
!python experiments/run_suite.py --device cuda --output-dir "$output_dir"
!python experiments/audit.py --output-dir "$output_dir"
!python experiments/analyze.py --output-dir "$output_dir"

## Xem kết quả
Bảng chính dùng final checkpoint, mean ± sample SD của 3 seed.


In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(Path(output_dir) / "tables/summary.csv"))
display(Image(filename=str(Path(output_dir) / "figures/04_final_comparison.png")))